## Example 5.6-1

In [2]:
ALPHA_IDX = 1
THETA_IDX = 2
Q_IDX = 3
H_IDX = 4

In [3]:
import sys
sys.path.append('../')
from tools.lin_transport import get_lin_transport
import control as ct
import numpy as np
import matplotlib.pyplot as plt
%matplotlib widget

In [4]:
RTOD = 57.296
# Constants
Vt = 250  # ft/s
alpha = 2.0/RTOD  # rad
gamma = -2.5/RTOD  # rad
theta = alpha + gamma  # rad
altitude = 36.37 # ft
x_Earth = 0  # ft
X0 = [Vt, alpha, theta, 0, altitude, x_Earth]
thtl = 0.2  # throttle 0-1
elev = -2.0  # deg
U0 = [thtl, elev]

sys = get_lin_transport(X0, U0, gamma, land=True)

[-4.03573880e-04  3.13308853e-01 -1.48913355e+01]
Trim results:
Cost = 1.357711294554987e-12
Keas (knots): 148.27
Alpha (deg): -0.02
Theta (deg): -2.52
Throttle (0-1): 0.31
Elevator (deg): -14.89


In [5]:
keep = [ALPHA_IDX, THETA_IDX, Q_IDX, H_IDX]
ap = sys.A[keep][:,keep]
bp = sys.B[keep][:,[1]]
cp = sys.C[[Q_IDX, THETA_IDX, H_IDX]][:,keep]
dp = sys.D[[Q_IDX, THETA_IDX, H_IDX]][:,[1]]
plant = ct.ss(ap, bp, cp, dp)
plant

<LinearIOSystem:sys[3]:['u[0]']->['y[0]', 'y[1]', 'y[2]']>

In [6]:
sysa = ct.ss([-10],[10],[-1],[0])
sys1 = ct.series(sysa, plant)
sys1

<LinearICSystem:sys[5]:['u[0]']->['y[0]', 'y[1]', 'y[2]']>

#### Flare Model to Follow

In [7]:
sysm = ct.ss([-0.3],[1],[1],[0])
ct.ss2tf(sysm)

TransferFunction(array([1.]), array([1. , 0.3]))

##### Build Compensator Dynamics

In [8]:
sysi = ct.ss([0],[1],[1],[0])
sysle = ct.ss([-10],[1],[1],[0])
syscomp = ct.series(sysi, sysle)
ct.ss2tf(syscomp)

TransferFunction(array([1.]), array([ 1., 10.,  0.]))

In [9]:
sys2 = ct.series(sysm - sys1[2,0], syscomp)
sys2

<LinearICSystem:sys[20]:['u[0]']->['y[0]']>

In [10]:
# State vector indices
DE_IDX = 0
ALPHA_IDX = 1
THETA_IDX = 2
Q_IDX = 3
H_IDX = 4
HBAR_IDX = 5
EPS_IDX = 6
X1_IDX = 7

# Output vector indices
X1_O_IDX = 0
EPS_O_IDX = 1
ERR_O_IDX = 2
Q_O_IDX = 3
THETA_O_IDX = 4
HBAR_O_IDX = 5

#### Build B and C Matrices

In [11]:
B = sys2.B
B[HBAR_IDX,0] = 0
C = np.zeros((6, sys2.A.shape[0]))
C[X1_O_IDX, X1_IDX] = 1
C[EPS_O_IDX, EPS_IDX] = 1
C[ERR_O_IDX, :] = [0, 0, 0, 0, -1, 1, 0, 0]
C[Q_O_IDX, Q_IDX] = 57.3
C[THETA_O_IDX, THETA_IDX] = 57.3
C[HBAR_O_IDX, HBAR_IDX] = 1
sys3 = ct.ss(sys2.A, B, C, np.zeros((6,1)))
sys3[ERR_O_IDX, 0]

StateSpace(array([[-1.00000000e+01,  0.00000000e+00,  0.00000000e+00,
         0.00000000e+00,  0.00000000e+00,  0.00000000e+00,
         0.00000000e+00,  0.00000000e+00],
       [ 0.00000000e+00, -6.45624185e-01,  5.61292117e-03,
         1.00000000e+00,  3.74266922e-06,  0.00000000e+00,
         0.00000000e+00,  0.00000000e+00],
       [ 0.00000000e+00,  0.00000000e+00,  0.00000000e+00,
         1.00000000e+00,  0.00000000e+00,  0.00000000e+00,
         0.00000000e+00,  0.00000000e+00],
       [ 1.09964713e-02, -7.73141910e-01, -8.10105534e-04,
        -5.29205180e-01, -3.15488072e-07,  0.00000000e+00,
         0.00000000e+00,  0.00000000e+00],
       [ 0.00000000e+00, -2.49762057e+02,  2.49762057e+02,
         0.00000000e+00,  0.00000000e+00,  0.00000000e+00,
         0.00000000e+00,  0.00000000e+00],
       [ 0.00000000e+00,  0.00000000e+00,  0.00000000e+00,
         0.00000000e+00,  0.00000000e+00, -3.00000000e-01,
         0.00000000e+00,  0.00000000e+00],
       [ 0.00000000e+00

In [12]:

v = ct.damp(sys3)

    Eigenvalue (pole)       Damping     Frequency
                  -10             1            10
                    0             1             0
 -0.002044  +0.02631j       0.07744       0.02639
 -0.002044  -0.02631j       0.07744       0.02639
   -0.5854   +0.8766j        0.5553         1.054
   -0.5854   -0.8766j        0.5553         1.054
                  -10             1            10
                 -0.3             1           0.3


c:\git\StevensAndLewis\.venv\lib\site-packages\control\lti.py:118: RuntimeWarning: invalid value encountered in divide
  zeta = -real(splane_poles)/wn


In [13]:
G = np.zeros((sys3.A.shape[0], 1))
G[HBAR_IDX, 0] = 1
F = np.zeros((sys3.C.shape[0], 1))
H = np.zeros((sys3.A.shape[0], 1))
H[H_IDX, 0] = -1
H[HBAR_IDX, 0] = 1

In [14]:
P_base = abs(H @ H.T)
print('P =\n', P_base)

Q = np.array([[0.001]])
print('Q =\n', Q)

R = np.array([[1]])  # control effort weight
print('R =\n', R)

P =
 [[0. 0. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 1. 1. 0. 0.]
 [0. 0. 0. 0. 1. 1. 0. 0.]
 [0. 0. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 0. 0.]]
Q =
 [[0.001]]
R =
 [[1]]


In [15]:
import tools.control_tools as tools
K_0 = np.zeros([sys3.ninputs, sys3.noutputs])

K_0[0, X1_O_IDX] = 500
K_0[0, EPS_O_IDX] = -10
K_0[0, ERR_O_IDX] = 6
K_0[0, Q_O_IDX] = -1
K_0[0, THETA_O_IDX] = -1
K_0[0, HBAR_O_IDX] = -1

Q_eff = sys3.C.T @ K_0.T @ R @ K_0 @ sys3.C + Q
vals = np.linalg.eigvalsh(Q_eff)
vals

K_opt = tools.LQTrackerTime(sys3, G, F, P_base, Q, R, K_0)

J_opt = 1169645271.0423408
K_opt =
 [[552.0829071  -18.77421623   0.61962543   2.28612523   1.62862718
    7.97692163]]


In [ ]:
K_opt = np.array([[593.4, -59.3, 6.154, -0.56, -1, -0.01852]])
syscl = ct.ss(sys3.A - sys3.B @ K_opt @ sys3.C, sys3.B, sys3.C, sys3.D)

In [ ]:
v = ct.damp(syscl)

    Eigenvalue (pole)       Damping     Frequency
                  -10             1            10
                    0             1             0
               -9.933             1         9.933
   -0.8024   +0.6853j        0.7604         1.055
   -0.8024   -0.6853j        0.7604         1.055
               0.3627             1        0.3627
            0.0004609             1     0.0004609
                 -0.3             1           0.3


c:\git\StevensAndLewis\.venv\lib\site-packages\control\lti.py:118: RuntimeWarning: invalid value encountered in divide
  zeta = -real(splane_poles)/wn


: 